# Example 1: Steady state heat transfer in 1D

This notebook walks through the script [`eg_1.py`](eg_1.py)
one piece at a time. See [`README.md`](README.md) for the full problem statement.

## The problem

We solve steady-state heat conduction in a rod occupying the interval $\Omega = (0, 1)$:

$$ -\nabla \cdot (k\,\nabla T) = 0 \quad \text{in } \Omega, \qquad T(0) = 100, \quad T(1) = 0, $$

with constant thermal conductivity $k = 0.1$ and no heat source.

The exact answer is the straight line $T(x) = 100\,(1 - x)$ with a constant flux
$q = -k\,\tfrac{dT}{dx} = 10$. We will recover both numerically.

## The anatomy of a DOLFINx script

Almost every DOLFINx simulation is assembled from the same seven ingredients. We go
through them in order:

1. **Backends**: MPI and PETSc
2. **Mesh**: the discretised domain
3. **Function space and functions**: where the solution lives
4. **Boundary conditions**
5. **Variational (weak) form**
6. **Solver**
7. **Solve and post-process**

## 1. Backends

DOLFINx is built on two libraries that do the heavy lifting:

- **`mpi4py` (MPI)** handles parallelism. Even when you run on a single core, the mesh
  and functions need to know about the *communicator* `MPI.COMM_WORLD`.
- **`petsc4py` (PETSc)** provides the data structures (vectors, matrices) and the
  numerical solvers (Newton, linear solvers, preconditioners).

We also import the DOLFINx pieces themselves:

- **`basix`**: defines finite elements,
- **`ufl`**: the language for writing variational forms,
- **`dolfinx.fem`**: function spaces, functions and boundary conditions.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc

import dolfinx
import basix
import numpy as np
from dolfinx import fem
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.mesh import create_mesh
import ufl

from dolfinx import plot
import pyvista

## 2. The mesh

The mesh is the discretised version of our domain $\Omega = (0,1)$. We split the
interval into 99 small cells (elements) defined by 100 points.

A mesh is built from three things:

- **the points** (`mesh_points`): the coordinates of the nodes,
- **the cells** (`cells`): which points are joined to form each element. In 1D each
  cell is an interval connecting two consecutive points,
- **the element description** (`domain`): here a degree-1 Lagrange element on an
  `"interval"` cell, living in a 1D geometry (`gdim = 1`).

In higher dimensions you would normally read a mesh from a file or use a built-in
helper like `dolfinx.mesh.create_unit_square`. Building it by hand here makes the
ingredients explicit.

In [ ]:
# 100 evenly spaced points from 0 to 1
indices = np.linspace(0, 1, num=100)

# element: degree-1 Lagrange on an interval, embedded in 1D space
gdim, shape, degree = 1, "interval", 1
domain = ufl.Mesh(basix.ufl.element("Lagrange", shape, degree, shape=(gdim,)))

# points need shape (n_points, gdim)
mesh_points = np.reshape(indices, (len(indices), 1))

# each cell connects point i to point i+1
indexes = np.arange(mesh_points.shape[0])
cells = np.stack((indexes[:-1], indexes[1:]), axis=-1)

my_mesh = create_mesh(comm=MPI.COMM_WORLD, cells=cells, x=mesh_points, e=domain)
fdim = my_mesh.topology.dim - 1

print(f"Mesh dimension: {my_mesh.topology.dim}")
print(f"Number of cells: {cells.shape[0]}")

## 3. Function space and functions

A **function space** `V` is the set of functions we allow our solution to be. Here we
use continuous, piecewise-linear (`"Lagrange"`, degree 1) functions. The unknown temperature will be a weighted sum of these.

We then create two objects living in that space:

- **`u`**: a `fem.Function`. This holds the *unknown* we solve for (its values are
  filled in when we call the solver).
- **`v`**: a `ufl.TestFunction`. This is the *test function* used to write the weak
  form below. It is symbolic; it never holds numbers.

In [ ]:
V = fem.functionspace(my_mesh, ("Lagrange", 1))
u = fem.Function(V)        # the unknown temperature T
v = ufl.TestFunction(V)    # test function

print(f"Degrees of freedom: {V.dofmap.index_map.size_global}")

## 4. Boundary conditions

We impose **Dirichlet** (fixed-value) conditions at both ends.

This is a two-step process:

1. **Find the degrees of freedom** on the boundary. `locate_dofs_geometrical` returns
   the dofs whose coordinates satisfy a condition: here `x = 0` for the left end and
   `x = 1` for the right end.
2. **Attach a value** to them with `fem.dirichletbc`. We pin the left end to `100` and
   the right end to `0`.

In [ ]:
dofs_L = fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], 0))
dofs_R = fem.locate_dofs_geometrical(V, lambda x: np.isclose(x[0], indices[-1]))

bc_left = fem.dirichletbc(fem.Constant(my_mesh, PETSc.ScalarType(100)), dofs_L, V)
bc_right = fem.dirichletbc(fem.Constant(my_mesh, PETSc.ScalarType(0)), dofs_R, V)
bcs = [bc_left, bc_right]

## 5. The variational (weak) form

DOLFINx does not solve the PDE in its *strong form* directly. Instead it solves the
**weak form**: multiply the equation by the test function $v$, integrate over the
domain, and integrate by parts.

Starting from $-\nabla\cdot(k\nabla T) = 0$ and multiplying by $v$:

$$ -\int_\Omega \nabla\cdot(k\nabla T)\, v \; dx = 0. $$

Integrating by parts (the boundary term vanishes because $v = 0$ where the value of
$T$ is fixed) gives the form we actually implement:

$$ F(u; v) = \int_\Omega k\,\nabla u \cdot \nabla v \; dx = 0. $$

In UFL this reads almost exactly like the maths. `ufl.dx` is the integral over the
whole domain.

In [ ]:
k = 0.1  # thermal conductivity
F = ufl.dot(k * ufl.grad(u), ufl.grad(v)) * ufl.dx

## 6. The solver

We hand the form `F`, the unknown `u` and the boundary conditions to a
`NonlinearProblem`, which uses PETSc's SNES (Newton) solver. Even though *this*
problem is linear, using the nonlinear interface means the exact same code structure
works for the nonlinear problems later in the fellowship (trapping, multi-species...).

The `petsc_options` dictionary configures the solver:

- `snes_type = newtonls`: Newton's method with line search,
- the `*_atol`, `*_rtol`, `*_stol`, `*_max_it` entries set the convergence tolerances,
- `ksp_type = preonly` + `pc_type = lu` with **MUMPS**: solve each linear step with a
  direct LU factorisation (robust and exact for small problems).

The little loop at the end clears these options out of the global PETSc options
database once the solver has read them, so they don't leak into other solvers.

In [ ]:
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_stol": np.sqrt(np.finfo(dolfinx.default_real_type).eps) * 1e-2,
    "snes_atol": 1e-10,
    "snes_rtol": 1e-10,
    "snes_max_it": 30,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
solver = NonlinearProblem(
    F,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="festim_solver",
)

# clean the options out of the global PETSc database
snes = solver.solver
prefix = snes.getOptionsPrefix()
opts = PETSc.Options()
for key in petsc_options.keys():
    del opts[f"{prefix}{key}"]

## 7. Solve

`solver.solve()` runs Newton's method and fills in the values of `u`.

In [ ]:
solver.solve()
print("Solved. Min/max temperature:", u.x.array.min(), u.x.array.max())

## 8. Post-processing

### Visualise the temperature

We use **PyVista** to draw the solution. The same few lines appear, wrapped in a
`plot_function` helper, at the top of [`eg_1.py`](eg_1.py); here we spell them out so
you can see what each step does.

> If the interactive PyVista window does not show up inside Jupyter, you may need a
> backend such as `trame`, or you can simply plot the raw arrays with the cell below.

In [ ]:
topology, cell_types, geometry = plot.vtk_mesh(V)
grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)
grid.point_data["T"] = u.x.array.real
grid.set_active_scalars("T")

plotter = pyvista.Plotter()
plotter.add_mesh(grid, cmap="viridis", show_edges=False)
plotter.show()

### Plot the raw values and check against the analytical solution

Because we used degree-1 elements, the degrees of freedom *are* the nodal temperature
values. We can pull them straight out of `u.x.array`. We sort them by their
coordinate (the dof ordering is not guaranteed to be left-to-right) and compare with
the exact line $T(x) = 100(1-x)$.

In [ ]:
x_dofs = V.tabulate_dof_coordinates()[:, 0]
order = np.argsort(x_dofs)
x_sorted = x_dofs[order]
T_sorted = u.x.array[order]

T_exact = 100 * (1 - x_sorted)
max_error = np.max(np.abs(T_sorted - T_exact))
print(f"Max error vs analytical solution: {max_error:.2e}")

### Evaluate the heat flux

The quantity an engineer usually cares about is the **heat flux**
$q = -k\,\dfrac{dT}{dx}$.

For our linear solution the flux is constant. We can estimate $dT/dx$ from the nodal
values with a finite difference and multiply by $-k$. The result should be very close
to the analytical value of `10`.

In [ ]:
dTdx = np.gradient(T_sorted, x_sorted)
flux = -k * dTdx

x_eval = 0.5
i = np.argmin(np.abs(x_sorted - x_eval))
print(f"Numerical flux at x = {x_sorted[i]:.3f}: {flux[i]:.4f}")
print(f"Analytical flux:                {-0.1 * -100:.4f}")

## Recap

You have now seen every ingredient of a DOLFINx simulation:
**backends → mesh → function space → boundary conditions → weak form → solver →
solve and post-process**.

The next examples reuse this exact skeleton and add one new concept at a time
(time stepping, extra fields, nonlinear source terms). If you can read this notebook,
you can read the rest.